# Results Analysis & Ablation Study
**Master Thesis | Sugimoto Lab**

This notebook produces all the tables and figures for the thesis:
1. Comparison table (all methods)
2. Ablation study (labeled ratio)
3. Failure case analysis
4. Visual overlays on CT slices

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os
REPO = '/content/drive/MyDrive/tesi'
os.chdir(REPO)
sys.path.insert(0, REPO)
!pip install monai[all] nibabel scipy seaborn --quiet

## 1. Load Results

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.2)

# Load aggregated results saved by 06_compare_methods.py
RESULTS_DIR = f'{REPO}/results'

with open(f'{RESULTS_DIR}/aggregated_results.json') as f:
    agg = json.load(f)

with open(f'{RESULTS_DIR}/per_case_results.json') as f:
    per_case = json.load(f)

print('Methods:', list(agg.keys()))

## 2. Comparison Table (LaTeX-ready)

In [ ]:
METRICS = ['tumor_dice', 'tumor_hd95', 'tumor_vol_error', 'pancreas_dice']
LABELS  = ['Tumor Dice↑', 'Tumor HD95↓ (mm)', 'Vol Error↓ (ml)', 'Pancreas Dice↑']

rows = []
for method, results in agg.items():
    row = {'Method': method.replace('_', ' ').title()}
    for metric, label in zip(METRICS, LABELS):
        m = results.get(f'{metric}_mean', float('nan'))
        s = results.get(f'{metric}_std',  float('nan'))
        row[label] = f'{m:.3f} ± {s:.3f}'
    rows.append(row)

df = pd.DataFrame(rows).set_index('Method')
display(df)

# Export as LaTeX table for thesis
latex = df.to_latex(caption='Quantitative comparison of segmentation methods on Task07 Pancreas (fold 0).',
                     label='tab:comparison', escape=False)
print('\n--- LaTeX ---')
print(latex)

## 3. Ablation Study — Effect of Labeled Ratio

In [ ]:
# Manually fill this with your results after running experiments
# with labeled_ratio = 0.05, 0.10, 0.20, 1.0 (supervised)

ablation = {
    'labeled_pct': [5,    10,   20,   100],
    'tumor_dice':  [None, None, None, None],  # fill after experiments
    'tumor_std':   [None, None, None, None],
}

# Once filled, plot:
fig, ax = plt.subplots(figsize=(8, 5))
valid = [(x, y, e) for x, y, e in zip(ablation['labeled_pct'],
                                        ablation['tumor_dice'],
                                        ablation['tumor_std']) if y is not None]
if valid:
    xs, ys, es = zip(*valid)
    ax.errorbar(xs, ys, yerr=es, marker='o', linewidth=2, markersize=8,
                color='#2196F3', capsize=5, label='Semi-supervised (Mean Teacher)')
    ax.set_xlabel('Labeled Training Data (%)')
    ax.set_ylabel('Tumor Dice (mean ± std)')
    ax.set_title('Ablation: Annotation Efficiency Curve')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/figures/ablation_labeled_ratio.png', dpi=150)
    plt.show()
else:
    print('Fill in ablation dict after running experiments with different labeled_ratio values.')

## 4. Failure Case Analysis

In [ ]:
# Find the worst-performing cases for the best method
best_method = max(agg, key=lambda m: agg[m].get('tumor_dice_mean', 0))
cases = per_case.get(best_method, [])

if cases:
    df_cases = pd.DataFrame(cases).sort_values('tumor_dice')
    print(f'Method: {best_method}')
    print(f'Best 5 cases:')
    display(df_cases.tail(5)[['tumor_dice', 'tumor_hd95', 'tumor_vol_error']].round(3))
    print(f'\nWorst 5 cases (failure analysis):')
    display(df_cases.head(5)[['tumor_dice', 'tumor_hd95', 'tumor_vol_error']].round(3))

    # Scatter: vol_gt vs dice (do small tumors fail more?)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(df_cases['tumor_vol_gt'], df_cases['tumor_dice'],
               alpha=0.6, color='#2196F3', edgecolors='k', linewidths=0.5)
    ax.set_xlabel('Ground Truth Tumor Volume (ml)')
    ax.set_ylabel('Tumor Dice')
    ax.set_title('Dice vs. Tumor Size — Failure Analysis')
    ax.axvline(x=5, color='red', linestyle='--', alpha=0.5, label='5 ml threshold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/figures/failure_analysis_size_vs_dice.png', dpi=150)
    plt.show()
    print('Correlation (tumor size vs Dice):', df_cases[['tumor_vol_gt','tumor_dice']].corr().iloc[0,1].round(3))

## 5. Visual Overlay on CT Slice

In [ ]:
import nibabel as nib
import numpy as np

def show_segmentation_overlay(ct_path, pred_path, gt_path, slice_idx=None):
    """Show CT slice with predicted and ground truth segmentation overlay."""
    ct   = np.asarray(nib.load(ct_path).dataobj)
    pred = np.asarray(nib.load(pred_path).dataobj)
    gt   = np.asarray(nib.load(gt_path).dataobj)

    # Find most informative slice (most tumor voxels)
    if slice_idx is None:
        tumor_per_slice = (gt == 2).sum(axis=(0, 1))
        slice_idx = int(np.argmax(tumor_per_slice)) if tumor_per_slice.max() > 0 else ct.shape[2] // 2

    ct_slice   = ct[:, :, slice_idx]
    pred_slice = pred[:, :, slice_idx]
    gt_slice   = gt[:, :, slice_idx]

    # Clip CT to pancreas window
    ct_display = np.clip(ct_slice, -125, 275)
    ct_display = (ct_display - ct_display.min()) / (ct_display.max() - ct_display.min() + 1e-8)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, mask, title, cmap in zip(
        axes,
        [None, pred_slice, gt_slice],
        ['CT Image', 'Prediction', 'Ground Truth'],
        [None, 'jet', 'jet']
    ):
        ax.imshow(ct_display.T, cmap='gray', origin='lower')
        if mask is not None:
            overlay = np.ma.masked_where(mask == 0, mask)
            ax.imshow(overlay.T, cmap=cmap, alpha=0.5, vmin=1, vmax=2, origin='lower')
        ax.set_title(title, fontsize=12)
        ax.axis('off')

    plt.suptitle(f'Axial slice {slice_idx}', fontsize=10)
    plt.tight_layout()
    return fig

# Example (fill paths after running inference):
# fig = show_segmentation_overlay(
#     ct_path   = 'data/nnunet_raw/Dataset007_Pancreas/imagesTr/pancreas_001_0000.nii.gz',
#     pred_path = 'experiments/semisup/predictions/pancreas_001.nii.gz',
#     gt_path   = 'data/nnunet_raw/Dataset007_Pancreas/labelsTr/pancreas_001.nii.gz',
# )
# fig.savefig(f'{RESULTS_DIR}/figures/overlay_example.png', dpi=150)
print('Overlay function ready. Uncomment and fill paths after running inference.')